# 手写 DPO（Direct Preference Optimization）

## 1. 核心公式

DPO 损失函数：

$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}_{(x,\, y_w,\, y_l)} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)} \right) \right]$$

其中：
- $x$：prompt
- $y_w$：preferred response（人类偏好的回复，win）
- $y_l$：rejected response（人类不偏好的回复，lose）
- $\pi_\theta$：当前策略模型（要训练的模型）
- $\pi_{\text{ref}}$：参考模型（SFT 后的模型，参数冻结）
- $\beta$：温度参数，控制策略偏离参考模型的程度
- $\sigma$：Sigmoid 函数

## 2. 代码变量与公式元素对应

| 代码变量 | 公式符号 | 含义 | shape |
|----------|----------|------|-------|
| `policy_chosen_logps` | $\log \pi_\theta(y_w \mid x)$ | 策略模型给 preferred response 的序列级 log 概率 | `(batch,)` |
| `policy_rejected_logps` | $\log \pi_\theta(y_l \mid x)$ | 策略模型给 rejected response 的序列级 log 概率 | `(batch,)` |
| `ref_chosen_logps` | $\log \pi_{\text{ref}}(y_w \mid x)$ | 参考模型给 preferred response 的序列级 log 概率 | `(batch,)` |
| `ref_rejected_logps` | $\log \pi_{\text{ref}}(y_l \mid x)$ | 参考模型给 rejected response 的序列级 log 概率 | `(batch,)` |
| `beta` | $\beta$ | 温度参数 | scalar |
| `chosen_rewards` | $\beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)}$ | preferred response 的隐式奖励 | `(batch,)` |
| `rejected_rewards` | $\beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}$ | rejected response 的隐式奖励 | `(batch,)` |
| `logits` | $\text{chosen\_rewards} - \text{rejected\_rewards}$ | Sigmoid 的输入 | `(batch,)` |
| `loss` | $\mathcal{L}_{\text{DPO}}$ | DPO 损失 | scalar |

In [ ]:
import torch
import torch.nn.functional as F

## 3. 辅助函数：计算序列级 log 概率

语言模型输出的是每个位置上对整个 vocab 的 logits，我们需要：
1. `log_softmax` → 转成 log 概率（数值稳定，避免先 softmax 再 log）
2. `gather` → 取出每个位置上**真实 token** 的 log 概率
3. `sum` → 把 token 级 log 概率加起来 = 序列级 log 概率

$$\log \pi_\theta(y \mid x) = \sum_{t=1}^{T} \log p_\theta(y_t \mid x, y_{<t})$$

In [ ]:
def get_batch_logps(logits, labels, mask):
    """
    从模型输出的 logits 计算序列级 log 概率。
    
    Args:
        logits: 模型输出, shape (batch, seq_len, vocab_size)
        labels: 真实 token id, shape (batch, seq_len)
        mask:   只对 response 部分为 1（不算 prompt 部分）, shape (batch, seq_len)
    
    Returns:
        seq_logps: 每个样本的序列级 log 概率, shape (batch,)
    """
    # Step 1: logits → log 概率（沿 vocab 维度，也就是最后一纬，做 log_softmax）得到 log-likelihood
    log_probs = F.log_softmax(logits, dim=-1)  # (batch, seq_len, vocab_size)

    # Step 2: 用 gather 取出每个位置上真实 token 的 log 概率
    # gather(input, dim, index): 沿 dim 维度，用 index 指定的位置去 input 中取值
    # labels.unsqueeze(-1) 形状: (batch, seq_len, 1) — 在 vocab 维度上指定要取哪个位置
    # 结果形状: (batch, seq_len, 1) → squeeze 后 (batch, seq_len)
    per_token_logps = torch.gather(log_probs, dim=2, index=labels.unsqueeze(-1)).squeeze(-1)
    # per_token_logps[i][t] = log P(labels[i][t] | 前面的 tokens)

    # Step 3: 只对 response 部分求和（mask 掉 prompt 和 padding）
    # 序列概率 = 各 token 概率的乘积，取 log 后乘积变求和
    seq_logps = (per_token_logps * mask).sum(dim=-1)  # (batch,)

    return seq_logps

## 4. DPO 损失函数

把公式拆成三步理解：
1. 计算隐式奖励：$r(x, y) = \beta \cdot (\log \pi_\theta(y|x) - \log \pi_{\text{ref}}(y|x))$
2. 比较 chosen 和 rejected 的奖励差：$\Delta r = r(x, y_w) - r(x, y_l)$
3. 过 Sigmoid 取负 log：$\mathcal{L} = -\log \sigma(\Delta r)$（即负对数似然）

In [ ]:
def dpo_loss(policy_chosen_logps, policy_rejected_logps, ref_chosen_logps, ref_rejected_logps, beta=0.1):
    """
    计算 DPO 损失。
    
    所有输入都是序列级 log 概率（由 get_batch_logps 计算得到），shape: (batch,)
    """
    # 公式: β * (log π_θ(y|x) - log π_ref(y|x)) = β * log(π_θ/π_ref)
    # 含义: 策略模型相对参考模型的 "隐式奖励"，也就是策略模型和reference模型生成好回复对数概率比
    chosen_rewards = beta * (policy_chosen_logps - ref_chosen_logps)      # (batch,)
    rejected_rewards = beta * (policy_rejected_logps - ref_rejected_logps)  # (batch,)

    # 公式: Δr = r(x, y_w) - r(x, y_l)
    # 我们希望 chosen 的隐式奖励 > rejected 的隐式奖励
    logits = chosen_rewards - rejected_rewards  # (batch,)

    # 公式: L = -log σ(Δr)，即负对数似然（Negative Log Likelihood）
    # 当 Δr > 0（chosen 奖励 > rejected），σ(Δr) 接近 1，loss 接近 0 ✓
    # 当 Δr < 0（chosen 奖励 < rejected），σ(Δr) 接近 0，loss 很大 ✗
    loss = -F.logsigmoid(logits).mean()  # scalar，对 batch 取平均

    # ---------- 以下是用于监控训练的指标，不参与梯度计算 ----------
    chosen_reward = chosen_rewards.detach().mean()    # 平均 chosen 隐式奖励
    rejected_reward = rejected_rewards.detach().mean()  # 平均 rejected 隐式奖励
    accuracy = (logits > 0).float().mean()  # 模型是否正确地给 chosen 更高的奖励

    return loss, chosen_reward, rejected_reward, accuracy

## 5. 简单测试

In [ ]:
# 模拟数据：batch_size=4
batch_size = 4

# 序列级 log 概率（负数，因为 log(概率) < 0）
policy_chosen_logps = torch.tensor([-5.0, -4.0, -6.0, -3.5])   # π_θ 给 chosen 的 log 概率
policy_rejected_logps = torch.tensor([-8.0, -7.0, -5.5, -9.0]) # π_θ 给 rejected 的 log 概率
ref_chosen_logps = torch.tensor([-5.5, -4.5, -6.5, -4.0])      # π_ref 给 chosen 的 log 概率
ref_rejected_logps = torch.tensor([-8.5, -7.5, -6.0, -9.5])    # π_ref 给 rejected 的 log 概率

loss, c_reward, r_reward, acc = dpo_loss(
    policy_chosen_logps, policy_rejected_logps,
    ref_chosen_logps, ref_rejected_logps,
    beta=0.1
)

print(f"DPO Loss:         {loss:.4f}")
print(f"Chosen Reward:    {c_reward:.4f}")
print(f"Rejected Reward:  {r_reward:.4f}")
print(f"Accuracy:         {acc:.2%}")

## 6. 常见面试追问

| 问题 | 要点 |
|------|------|
| DPO 和 RLHF(PPO) 的关系？ | DPO 是 RLHF 的等价闭式解，跳过了显式 Reward Model 和 PPO，直接用偏好数据优化策略 |
| β 越大/越小会怎样？ | β 越大 → 允许策略更大幅度偏离参考模型；β 越小 → 越保守，更接近参考模型 |
| 为什么需要 reference model？ | 起 KL 正则化的作用，防止策略模型为了拟合偏好数据而产生退化（如重复、胡说） |
| log 概率为什么是负数？ | 因为概率 ∈ (0,1)，log(概率) < 0 |
| 为什么用 logsigmoid 而不是 sigmoid + log？ | 数值稳定性，和 log_softmax 同理 |
| DPO 的缺点？ | 对偏好数据质量非常敏感；没有显式 reward 信号，难以监控 reward hacking |